# 显式资源管理

学习目标：能用同步和异步释放协议管理真实资源，并解释释放顺序及复合异常。

前置知识：块作用域、Symbol、异常传播、async/await、Node.js 文件 API。

适用版本：选修补充：超出 ECMAScript 2025 基线；使用 Node.js 24.11.0 的显式资源管理支持，.mjs 为 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/30-resource-management/。

1. [support.mjs](scripts/30-resource-management/support.mjs)：实际解析与内置协议存在性。
2. [sync-file.mjs](scripts/30-resource-management/sync-file.mjs)：真实文件的同步释放。
3. [async-file.mjs](scripts/30-resource-management/async-file.mjs)：真实 FileHandle 的异步关闭。
4. [stacks.mjs](scripts/30-resource-management/stacks.mjs)：use、adopt、defer 与 move。
5. [async-stack.mjs](scripts/30-resource-management/async-stack.mjs)：异步栈的逆序与等待。
6. [errors.mjs](scripts/30-resource-management/errors.mjs)：同步与异步复合错误断言。
7. [uncaught.mjs](scripts/30-resource-management/uncaught.mjs)：独立未捕获复合异常。

Step 1：运行语法与内置支持示例。

```bash
node scripts/30-resource-management/support.mjs
```

Step 2：运行纯同步文件释放。

```bash
node scripts/30-resource-management/sync-file.mjs
```

Step 3：运行异步文件释放。

```bash
node scripts/30-resource-management/async-file.mjs
```

Step 4：运行同步资源栈。

```bash
node scripts/30-resource-management/stacks.mjs
```

Step 5：运行异步资源栈。

```bash
node scripts/30-resource-management/async-stack.mjs
```

Step 6：运行错误传播断言。

```bash
node scripts/30-resource-management/errors.mjs
```

Step 7：单独运行预期失败的复合异常。

```bash
node scripts/30-resource-management/uncaught.mjs
```

## 1 标准状态与资源所有权

显式资源管理（explicit resource management）在离开词法作用域时调用约定的释放方法。资源可以是文件句柄、监听器或其他需要确定结束的对象；垃圾回收只处理内存可达性，不能代替及时关闭这些资源。

本课基线是 ECMA-262 第 16 版，using 与相关内置对象不属于该基线。2026-09-23 核对的 TC39 完成提案列表已将此提案列为 Stage 4，表中预期出版年份为 2027；“提案完成”“规范出版年份”和“宿主已实现”是三件事。Node.js 24 发布说明已启用这一特性，本章固定在 24.11.0 直接运行，不加转译或补丁。

只检测 Symbol.dispose 等属性不能证明解析器识别 using；包含语法的独立文件需要真正解析执行。换到其他运行环境时，应分别检查语法、内置类以及具体资源是否实现释放协议。下面最小文件同时覆盖这三层中的前两层，后续用真实文件覆盖资源适配。

配套 [support.mjs](scripts/30-resource-management/support.mjs)：

```javascript
import assert from "node:assert/strict";
assert.equal(typeof Symbol.dispose, "symbol");
assert.equal(typeof Symbol.asyncDispose, "symbol");
assert.equal(typeof DisposableStack, "function");
assert.equal(typeof AsyncDisposableStack, "function");
assert.equal(typeof SuppressedError, "function");
const events = [];
{
  using sync = { [Symbol.dispose]() { events.push("sync"); } };
  await using asyncResource = { async [Symbol.asyncDispose]() { await Promise.resolve(); events.push("async"); } };
}
assert.deepEqual(events, ["async", "sync"]);
console.log("syntax and resource builtins available"); // → syntax and resource builtins available
```

## 2 using 与同步释放顺序

先把资源登记顺序列出，再反向检查离开范围时的清理次序。

using 声明不可重新赋值的词法绑定，并在初始化时登记 Symbol.dispose 方法；本章都把它放在明确花括号范围内。资源对象不因声明而不可变。离开范围时按登记的逆序释放，正常结束、return 和 throw 都会经过释放过程。

null 与 undefined 可作为空资源，其余普通值必须满足相应协议；不能把任意对象都写进 using。同步释放不等待返回的 Promise，异步操作应使用异步协议。using 不支持像普通 const 那样使用解构绑定。

下面文件不包含 await，通过 Node.js 同步 openSync/closeSync 管理真实文件描述符。保存旧描述符只用于验证关闭后的 EBADF 错误，业务中不应继续使用已释放句柄。

![资源按登记的逆序释放。同步 using：返回给调用者之前，先完成本作用域已登记的释放。](image/illustration/30-01-using-disposal-order.svg)

图示说明：依据 using 释放顺序自绘，图省略业务主体的写入操作；null 不登记释放动作，异常合并规则在后文另述。

在 writeNote 中找到 file、marker、absent 的声明，再用 events 检查 body 之后两次释放的顺序。

配套 [sync-file.mjs](scripts/30-resource-management/sync-file.mjs)：

```javascript
import { rm } from "node:fs/promises";
import assert from "node:assert/strict";
import { openSync, closeSync, writeFileSync, readFileSync, fstatSync, mkdtempSync } from "node:fs";
import { resolve, relative, isAbsolute } from "node:path";
class SyncFile {
  constructor(path, events) { this.fd = openSync(path, "w"); this.events = events; }
  [Symbol.dispose]() {
    // 释放后置空；重复释放不再关闭可能被系统重新分配的文件描述符。
    if (this.fd === undefined) return;
    closeSync(this.fd);
    this.fd = undefined;
    this.events.push("file closed");
  }
}
const root = import.meta.dirname;
const directory = mkdtempSync(resolve(root, "js-c-sync-"));
try {
  const events = [];
  const path = resolve(directory, "note.txt");
  let savedFd;
  // 函数作用域结束时按声明的逆序释放：先 marker，后 file。
  function writeNote() {
    using file = new SyncFile(path, events);
    using marker = { [Symbol.dispose]() { events.push("marker disposed"); } };
    // null 不注册释放动作；保留此行用于观察允许的空资源。
    using absent = null;
    savedFd = file.fd;
    writeFileSync(file.fd, "saved", "utf8");
    events.push("body");
    return "returned";
  }
  // 返回值已取得时资源也已释放：读回文件，并检查旧描述符已关闭。
  assert.equal(writeNote(), "returned");
  assert.deepEqual(events, ["body", "marker disposed", "file closed"]);
  assert.equal(readFileSync(path, "utf8"), "saved");
  assert.throws(() => fstatSync(savedFd), { code: "EBADF" });
  console.log(events.join(",")); // → body,marker disposed,file closed
} finally {
  // 文件释放与临时目录清理是两层职责；失败时仍删除本次目录。
  const child = relative(root, directory);
  assert.ok(child.startsWith("js-c-sync-") && !isAbsolute(child) && !child.includes(".."));
  await rm(directory, { recursive: true });
}
console.log("synchronous file cleaned"); // → synchronous file cleaned
```

## 3 await using 与异步释放

await using 在释放时优先调用 Symbol.asyncDispose 并等待其结果；若没有异步方法，可回退到同步释放方法。它只能出现在允许相应异步语法的位置，本例使用 ES 模块顶层和 async 函数。

await using 不会自动等待右侧初始化表达式。异步创建资源要写 await using handle = await open(...)，右侧 await 等待打开，左侧 await using 等待关闭。只把 Promise 对象绑定为资源并不能获得文件句柄的释放能力。

Node.js 24.11.0 的 FileHandle 实现 Symbol.asyncDispose，等价于调用并等待关闭。以下示例等待写入、离开作用域关闭，再回读文件并核对已保存句柄不能继续 stat；退出文件实验后删除目录。

配套 [async-file.mjs](scripts/30-resource-management/async-file.mjs)：

```javascript
import assert from "node:assert/strict";
import { open, readFile, mkdtemp, rm } from "node:fs/promises";
import { resolve, relative, isAbsolute } from "node:path";
const root = import.meta.dirname;
const directory = await mkdtemp(resolve(root, "js-c-async-"));
try {
  const path = resolve(directory, "note.txt");
  let savedHandle;
  // 这个块界定文件寿命；写入与离开块时的异步关闭都要等待。
  {
    await using file = await open(path, "w");
    savedHandle = file;
    await file.writeFile("async saved", "utf8");
  }
  // 块外再读回文件，并用保留的句柄观察已关闭状态。
  assert.equal(await readFile(path, "utf8"), "async saved");
  await assert.rejects(savedHandle.stat(), { code: "EBADF" });
  console.log("write awaited and handle closed"); // → write awaited and handle closed
  // 对照：没有 asyncDispose 时，await using 也可以采用同步 dispose。
  const events = [];
  {
    await using fallback = { [Symbol.dispose]() { events.push("sync fallback"); } };
  }
  assert.deepEqual(events, ["sync fallback"]);
} finally {
  // 句柄关闭后清理本次目录；这里的 finally 不吞掉原始异常。
  const child = relative(root, directory);
  assert.ok(child.startsWith("js-c-async-") && !isAbsolute(child) && !child.includes(".."));
  await rm(directory, { recursive: true });
}
console.log("asynchronous file cleaned"); // → asynchronous file cleaned
```

## 4 用 DisposableStack 组合多个清理动作

当资源数量在运行时决定，DisposableStack 可把多项释放登记成一个栈。use 接收符合协议的资源并返回它；adopt 接收普通值和释放该值的函数；defer 登记不接收资源值的清理函数。栈本身实现 Symbol.dispose，因此可以交给 using 管理。

dispose 按逆序执行清理；重复 dispose 不再次运行这些动作。move 把登记项转移给新栈，让原栈进入已释放状态，常用于初始化完成后转交资源所有权。被转移或已释放的栈不能再登记新动作；它并没有把资源对象复制一份。

示例中的动作只记录释放顺序。把函数换成关闭已拥有的句柄时，还应处理登记之前创建资源失败的边界，避免资源成功创建却未被登记。

配套 [stacks.mjs](scripts/30-resource-management/stacks.mjs)：

```javascript
import assert from "node:assert/strict";
const events = [];
const setup = new DisposableStack();
setup.use({ [Symbol.dispose]() { events.push("use"); } });
assert.equal(setup.adopt("fd-7", (value) => events.push(`adopt:${value}`)), "fd-7");
setup.defer(() => events.push("defer"));
const owned = setup.move();
assert.equal(setup.disposed, true);
setup.dispose();
assert.equal(events.length, 0);
assert.throws(() => setup.defer(() => {}), ReferenceError);
{
  using scope = owned;
}
owned.dispose();
assert.deepEqual(events, ["defer", "adopt:fd-7", "use"]);
console.log(events.join(",")); // → defer,adopt:fd-7,use
```

## 5 AsyncDisposableStack 等待每次异步释放

AsyncDisposableStack 对应异步生命周期：use 可登记异步或同步资源，adopt/defer 可登记返回 Promise 的释放动作，disposeAsync 返回完成所有释放的 Promise。把栈交给 await using 可以把等待责任保持在作用域边界。

清理依次逆序执行，不是把所有清理函数同时扔进 Promise.all；后创建的资源可能依赖先创建的资源，因此顺序是所有权设计的一部分。即使有一项失败，仍继续处理其他登记项，并按释放算法传播错误。

配套 [async-stack.mjs](scripts/30-resource-management/async-stack.mjs)：

```javascript
import assert from "node:assert/strict";
const events = [];
const stack = new AsyncDisposableStack();
// 依次登记资源、带值的清理和无参数清理；释放时顺序反过来。
stack.use({ async [Symbol.asyncDispose]() {
  events.push("resource start");
  await Promise.resolve();
  events.push("resource end");
} });
stack.adopt("token", async (value) => {
  events.push(`adopt:${value}`);
  await Promise.resolve();
});
stack.defer(async () => {
  events.push("defer start");
  await Promise.resolve();
  events.push("defer end");
});
// 等待整个释放链，再次调用用于观察不会重复执行清理。
await stack.disposeAsync();
await stack.disposeAsync();
assert.equal(stack.disposed, true);
assert.deepEqual(events, ["defer start", "defer end", "adopt:token", "resource start", "resource end"]);
console.log(events.join(",")); // → defer start,defer end,adopt:token,resource start,resource end
```

## 6 释放失败与 SuppressedError

只有主体抛错时，释放成功后继续抛出主体错误；只有释放失败时，传播该释放错误。若主体或先前释放已经产生异常，而后续释放再次失败，SuppressedError 把两者连接起来：error 保存本次释放错误，suppressed 保存先前异常。多个释放失败会形成嵌套结构，不是 AggregateError 的 errors 数组。

这种机制保留被后续错误遮盖的信息。它不同于普通 try/finally 中一个 throw 直接覆盖已有异常。异步释放拒绝也进入相同的合并逻辑；调用者仍须 await 整个工作，再检查最终异常链。

下面既断言纯主体错误、单次释放错误，也检查主体与两次同步释放形成的嵌套，以及一次异步释放拒绝。Node.js 默认未捕获诊断不一定展开 error 与 suppressed；最后独立文件显式输出这两个原因后重新抛出，进程仍以失败状态退出。

配套 [errors.mjs](scripts/30-resource-management/errors.mjs)：

```javascript
import assert from "node:assert/strict";
// 单独比较正文失败与仅释放失败，确认原始错误直接传播。
assert.throws(() => {
  using item = { [Symbol.dispose]() {} };
  throw new Error("body only");
}, /body only/);
// 单独比较正文失败与仅释放失败，确认原始错误直接传播。
assert.throws(() => {
  using item = { [Symbol.dispose]() { throw new Error("dispose only"); } };
}, /dispose only/);

// 两个释放器都故意抛错：记录释放次序，再从外向内检查复合错误。
const released = [];
assert.throws(() => {
  using first = { [Symbol.dispose]() { released.push("first"); throw new Error("first cleanup"); } };
  using second = { [Symbol.dispose]() { released.push("second"); throw new Error("second cleanup"); } };
  throw new Error("body");
}, (error) => {
  // error 是较晚出现的释放错误；suppressed 保留此前正在传播的错误。
  assert.ok(error instanceof SuppressedError);
  assert.equal(error.error.message, "first cleanup");
  assert.ok(error.suppressed instanceof SuppressedError);
  assert.equal(error.suppressed.error.message, "second cleanup");
  assert.equal(error.suppressed.suppressed.message, "body");
  return true;
});
assert.deepEqual(released, ["second", "first"]);
console.log("sync error chain first/second/body"); // → sync error chain first/second/body

// 异步资源用相同的对照：await rejects 等待正文和释放都结束。
async function failAsync() {
  await using item = { async [Symbol.asyncDispose]() {
    await Promise.resolve();
    throw new Error("async cleanup");
  } };
  throw new Error("async body");
}
await assert.rejects(failAsync(), (error) => {
  // error 是较晚出现的释放错误；suppressed 保留此前正在传播的错误。
  assert.ok(error instanceof SuppressedError);
  assert.equal(error.error.message, "async cleanup");
  assert.equal(error.suppressed.message, "async body");
  return true;
});
assert.throws(() => { using invalid = {}; }, TypeError);
console.log("async error chain cleanup/body"); // → async error chain cleanup/body
```

配套 [uncaught.mjs](scripts/30-resource-management/uncaught.mjs)：

```javascript
try {
  using item = { [Symbol.dispose]() { throw new Error("DISPOSE_FAILURE"); } };
  throw new Error("BODY_FAILURE");
} catch (error) {
  console.error(error.error.message, error.suppressed.message); // stderr 首行：DISPOSE_FAILURE BODY_FAILURE
  throw error;
}
// → Node.js 退出 1；诊断同时含 SuppressedError、DISPOSE_FAILURE、BODY_FAILURE
```

## 本章小结

- using 和 await using 把释放责任绑定到明确作用域；初始化等待与释放等待是两个位置。
- 栈按登记逆序释放，move 转移责任，不复制资源。
- SuppressedError 保留主体与清理的复合失败，异步释放仍须等待并处理拒绝。

## 练习

1. 交换同步文件与 marker 的声明顺序；标准：释放日志也交换，文件仍关闭且可以删除。
2. 给异步栈的最后一个 defer 加入拒绝；标准：它之前登记的资源仍释放，最终 disposeAsync 拒绝。
3. 去掉 errors.mjs 的主体错误，只保留两个释放错误；标准：最终 error 对应 first cleanup，suppressed 对应 second cleanup，不再含 body。

### 提示

1. 只交换 writeNote 中 file 与 marker 的两条声明，并同步 events 的期望顺序。
2. 把最后一个 defer 的结束点改为 throw new Error("defer failed")，用 await assert.rejects 观察第一次 disposeAsync，并更新日志期望。
3. 只删“两资源释放失败”组的 throw new Error("body")；不要删除其他独立对照组的主体错误。


### 参考解析

1. 声明改为 marker 在前、file 在后，释放日志是 body,file closed,marker disposed；文件关闭断言和目录清理仍成立。
2. 日志先出现 defer start，随后仍有 adopt:token、resource start、resource end；第一次 disposeAsync 拒绝 defer failed，第二次不再运行这些动作。若只是直接 await 后未捕获，脚本会在第一次拒绝处结束，无法打印后续断言结果。
3. 先抛 second cleanup，后抛 first cleanup，最终为 SuppressedError：error.message 是 first cleanup，suppressed.message 是 second cleanup。同步组对应断言和输出说明都须更新，其他对照组不变。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39 | [显式资源管理提案规范](https://tc39.es/proposal-explicit-resource-management/)，§2.1 GetDisposeMethod / DisposeResources、§7.2 using 与 await using、§11.1.4 SuppressedError、§12.3 DisposableStack、§12.4 AsyncDisposableStack：本文使用的释放与错误合并规则。 |
| GitHub / TC39 | [Finished Proposals](https://github.com/tc39/proposals/blob/main/finished-proposals.md)，Explicit Resource Management 行：2026-09-23 核对时 Stage 4，预期出版年份栏为 2027，属于超出本课 2025 基线的补充。 |
| Node.js | [fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[24.0.0 发布说明](https://nodejs.org/en/blog/release/v24.0.0)中的 enable explicit resource management；24.11.0 [FileHandle 的 Symbol.asyncDispose](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#filehandlesymbolasyncdispose)、同页 openSync、closeSync、fstatSync、mkdtemp 与 rm：真实文件资源的宿主支持。 |
